# Stage 2b — LSTM baseline

A deliberately simple, from-scratch 2-layer LSTM (see architecture discussion) trained via
SageMaker script mode -- unlike DeepAR, this isn't a built-in algorithm, so we bring our own
training script (`lstm_src/train.py`).

**Reuses Stage 2's data as-is**: `train.json`/`test.json` in `s3://.../ts-forecast-demo/deepar-v1/`
are model-agnostic (just per-client series), so there's no DuckDB step here at all -- also sidesteps
the DuckDB/pyarrow extension-type registry bug from Stage 2.

**Tags and results persistence from the start** this time, not retrofitted at the end: every job
is tagged, and `train.py` itself pushes a results JSON straight to S3 as soon as each job finishes.

**No real-time endpoint in this notebook** -- deliberate cut, not an oversight. That pattern was
already demonstrated with DeepAR; repeating it here adds cost without new learning value.

In [ ]:
%pip install -q "sagemaker<3" boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.pytorch import PyTorch
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

REGION = boto3.Session().region_name or "us-east-1"
BUCKET = "<your-bucket>"
SAGEMAKER_ROLE = "arn:aws:iam::<ACCOUNT_ID>:role/ts-forecast-demo-sagemaker-role"

DEEPAR_PREFIX = "ts-forecast-demo/deepar-v1"
LSTM_PREFIX = "ts-forecast-demo/lstm-v1"
RESULTS_PREFIX = "ts-forecast-demo/results/lstm-v1"  # per-job JSON, pushed directly by train.py
CODE_LOCATION = f"s3://{BUCKET}/{LSTM_PREFIX}/code"  # NOT the SageMaker default bucket -- our
                                                       # IAM policy is scoped to our own bucket only

train_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/train/"
test_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/test/"

CONTEXT_LENGTH = 168
PREDICTION_LENGTH = 168

TAGS = [
    {"Key": "Project", "Value": "ts-forecast-demo"},
    {"Key": "Stage", "Value": "2b-lstm-v1"},
    {"Key": "Model", "Value": "lstm"},
]

METRIC_DEFINITIONS = [{"Name": "validation:rmse", "Regex": "validation:rmse=([0-9.]+)"}]

session = sagemaker.Session()
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")

## 1. Baseline training job (CPU)

Sanity-check the script end to end -- data loading, model shapes, S3 results upload -- before
spending anything on HPO. Hyperparameter defaults here are exactly the architecture we settled on:
2 layers, hidden_size=128.

In [ ]:
baseline_hyperparameters = {
    "hidden-size": 128,
    "num-layers": 2,
    "embedding-dim": 16,
    "dropout": 0.1,
    "learning-rate": 1e-3,
    "epochs": 30,
    "batch-size": 64,
    "steps-per-epoch": 200,
    "context-length": CONTEXT_LENGTH,
    "prediction-length": PREDICTION_LENGTH,
    "s3-bucket": BUCKET,
    "s3-results-prefix": RESULTS_PREFIX,
}

baseline_estimator = PyTorch(
    entry_point="train.py",
    source_dir="lstm_src",
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    framework_version="2.1",
    py_version="py310",
    output_path=f"s3://{BUCKET}/{LSTM_PREFIX}/model-baseline",
    code_location=CODE_LOCATION,
    hyperparameters=baseline_hyperparameters,
    metric_definitions=METRIC_DEFINITIONS,
    tags=TAGS,
    sagemaker_session=session,
)

baseline_estimator.fit({"train": TrainingInput(train_s3), "test": TrainingInput(test_s3)})

## 2. Hyperparameter tuning (small budget, CPU)

Same budget discipline as DeepAR: 6 jobs, 2 parallel. Searching the knobs we were least sure about
from the architecture discussion -- hidden_size, num_layers, learning_rate, dropout.

In [ ]:
tuning_estimator = PyTorch(
    entry_point="train.py",
    source_dir="lstm_src",
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    framework_version="2.1",
    py_version="py310",
    output_path=f"s3://{BUCKET}/{LSTM_PREFIX}/model-hpo",
    code_location=CODE_LOCATION,
    hyperparameters={
        "embedding-dim": 16,
        "epochs": 30,
        "batch-size": 64,
        "steps-per-epoch": 200,
        "context-length": CONTEXT_LENGTH,
        "prediction-length": PREDICTION_LENGTH,
        "s3-bucket": BUCKET,
        "s3-results-prefix": RESULTS_PREFIX,
    },
    metric_definitions=METRIC_DEFINITIONS,
    tags=TAGS,
    sagemaker_session=session,
)

hyperparameter_ranges = {
    "hidden-size": IntegerParameter(64, 256),
    "num-layers": IntegerParameter(1, 2),
    "learning-rate": ContinuousParameter(1e-4, 1e-2),
    "dropout": ContinuousParameter(0.0, 0.3),
}

tuner = HyperparameterTuner(
    estimator=tuning_estimator,
    objective_metric_name="validation:rmse",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=METRIC_DEFINITIONS,
    max_jobs=6,
    max_parallel_jobs=2,
    tags=TAGS,
)

tuner.fit({"train": TrainingInput(train_s3), "test": TrainingInput(test_s3)})

In [ ]:
best_job = tuner.best_training_job()
best_hyperparameters = sagemaker.estimator.Estimator.attach(best_job, sagemaker_session=session).hyperparameters()
print(best_job)
print(best_hyperparameters)

## 3. GPU comparison run

Same best hyperparameters, ml.g4dn.xlarge instead of ml.c5.xlarge. Expectation to test empirically:
LSTM's sequential recurrence limits GPU parallelism far more than DeepAR's (also recurrent, but
smaller) or TFT's (attention, parallelizes over sequence positions) -- so the speedup here should
be more modest than a naive "GPU is always faster" intuition would suggest.

In [ ]:
gpu_hyperparameters = {k: v for k, v in best_hyperparameters.items() if not k.startswith("_")}
gpu_hyperparameters["s3-bucket"] = BUCKET
gpu_hyperparameters["s3-results-prefix"] = RESULTS_PREFIX

gpu_estimator = PyTorch(
    entry_point="train.py",
    source_dir="lstm_src",
    role=SAGEMAKER_ROLE,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    framework_version="2.1",
    py_version="py310",
    output_path=f"s3://{BUCKET}/{LSTM_PREFIX}/model-gpu",
    code_location=CODE_LOCATION,
    hyperparameters=gpu_hyperparameters,
    metric_definitions=METRIC_DEFINITIONS,
    tags=TAGS,
    sagemaker_session=session,
)

gpu_estimator.fit({"train": TrainingInput(train_s3), "test": TrainingInput(test_s3)})

## 4. Cost / instance-time comparison

In [ ]:
job_labels = {
    "baseline (CPU)": baseline_estimator.latest_training_job.name,
    "best HPO (CPU)": best_job,
    "comparison (GPU)": gpu_estimator.latest_training_job.name,
}

for name, job_name in job_labels.items():
    desc = sm.describe_training_job(TrainingJobName=job_name)
    print(f"{name:20s} instance={desc['ResourceConfig']['InstanceType']:15s} "
          f"billable_seconds={desc['TrainingTimeInSeconds']}")

## 5. Pull results (already in S3, pushed by train.py itself)

No batch transform needed -- `train.py` computed held-out-week RMSE and uploaded it directly during
each job. We just read those files back.

In [ ]:
def load_job_result(job_name):
    obj = s3.get_object(Bucket=BUCKET, Key=f"{RESULTS_PREFIX}/{job_name}.json")
    return json.loads(obj["Body"].read())

job_results = {name: load_job_result(job_name) for name, job_name in job_labels.items()}

for name, result in job_results.items():
    print(f"{name:20s} mean_rmse={result['mean_rmse']:.2f} median_rmse={result['median_rmse']:.2f} "
          f"train_seconds={result['train_seconds']:.1f}")

## 6. Persist the LSTM model card

Same location pattern as DeepAR's (`results/<model>-v1/model_card.json`), so the two are
trivially comparable side by side.

In [ ]:
model_card = {
    "model": "lstm_v1",
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "best_job": best_job,
    "best_hyperparameters": best_hyperparameters,
    "jobs": {
        name: {
            "job_name": job_name,
            "instance_type": sm.describe_training_job(TrainingJobName=job_name)["ResourceConfig"]["InstanceType"],
            "billable_seconds": sm.describe_training_job(TrainingJobName=job_name)["TrainingTimeInSeconds"],
            "mean_rmse": job_results[name]["mean_rmse"],
            "median_rmse": job_results[name]["median_rmse"],
        }
        for name, job_name in job_labels.items()
    },
}

s3.put_object(
    Bucket=BUCKET,
    Key="ts-forecast-demo/results/lstm-v1/model_card.json",
    Body=json.dumps(model_card, indent=2).encode(),
)
print("Persisted s3://%s/ts-forecast-demo/results/lstm-v1/model_card.json" % BUCKET)

## 7. DeepAR vs. LSTM, side by side

Both model cards live at the same path shape -- read DeepAR's back in for a direct comparison,
the first real payoff of persisting results consistently across notebooks.

In [ ]:
deepar_card = json.loads(
    s3.get_object(Bucket=BUCKET, Key="ts-forecast-demo/results/deepar-v1/model_card.json")["Body"].read()
)

print(f"{'model':10s} {'mean_rmse':>12s}")
print(f"{'deepar_v1':10s} {deepar_card['evaluation']['mean_rmse']:>12.2f}")
print(f"{'lstm_v1':10s} {job_results['best HPO (CPU)']['mean_rmse']:>12.2f}")